Read verdict file and print some sample


In [13]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
raw_text[100:]

Total number of character: 20480


'reat surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even through the prism of Hermia\'s tears I felt able to face the fact with equanimity. Poor Jack Gisburn

Create a token by spliting string by space and special character. trim empty spaces

In [14]:
import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(len(preprocessed))
print(preprocessed[:30])



4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


Let's now create a list of all unique tokens and sort
them alphabetically to create vocabulary

In [3]:
all_words = sorted(set(preprocessed))
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab_size = len(all_words)
print("vocab_size:", vocab_size)

vocab = {token:integer for integer,token in enumerate(all_words)}

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


vocab_size: 1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


when we want to convert the outputs of an LLM from numbers back into
text, we also need a way to turn token IDs into text. For this, we can create an inverse
version of the vocabulary that maps token IDs back to corresponding text tokens.
Let's implement a complete tokenizer class in Python with an encode method that splits
text into tokens and carries out the string-to-integer mapping to produce token IDs via the
vocabulary. In addition, we implement a decode method that carries out the reverse
integer-to-string mapping to convert the token IDs back into text

In [4]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab 
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids): 
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) #replace space before special char
        return text

Let's instantiate a new tokenizer object from the SimpleTokenizerV1 class and tokenize a
passage from Edith Wharton's short story

In [5]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable
pride."""
ids = tokenizer.encode(text)
print(ids)

print(tokenizer.decode(ids))

text1 = "Hello, do you like tea?"
ids = tokenizer.encode(text1)

print(ids)

print(tokenizer.decode(ids))



[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.
[1131, 5, 355, 1126, 628, 975, 10]
<|unk|>, do you like tea?


Byte pair encoding
The original BPE algorithm operates by iteratively replacing the most common contiguous sequences of characters in a target text with unused 'placeholder' bytes. The iteration ends when no sequences can be found, leaving the target text effectively compressed. Decompression can be performed by reversing this process, querying known placeholder terms against their corresponding denoted sequence, using a lookup table. In the original paper, this lookup table is encoded and stored alongside the compressed text.

Example
Suppose we are encoding the previous example of "aaabdaaabac", with a specified vocabulary size of 6, then it would first be encoded as "0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3". Then it would proceed as before, and obtain "4, 5, 2, 4, 5, 0, 3" with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5".

So far this is essentially the same as before. However, if we only had specified a vocabulary size of 5, then the process would stop at vocabulary "a=0, b=1, d=2, c=3, aa=4", so that the example would be encoded as "4, 0, 1, 2, 4, 0, 1, 0, 3". Conversely, if we had specified a vocabulary size of 8, then it would be encoded as "7, 6, 0, 3", with a vocabulary of "a=0, b=1, d=2, c=3, aa=4, ab=5, aaab=6, aaabd=7". This is not maximally compressed, because modified BPE does not aim for maximum compression. Instead, it aims for an encoding that is efficient and practical for language model training.[13]

In [6]:
pip install tiktoken


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

strings = tokenizer.decode(integers)
print(strings)




tiktoken version: 0.13.0
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


Create Datasets and DataLoader

In [8]:
pip install torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class DataSetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.data = []
        self.label = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunks = token_ids[i:i+max_length]
            target_chunks = token_ids[i+1:i+max_length + 1]
            self.data.append(torch.tensor(input_chunks))
            self.label.append(torch.tensor(target_chunks))
    
    def __len__(self): 
        return len(self.data)
    
    def __getitem__(self, idx): 
        return self.data[idx], self.label[idx]



/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Now create dataloader

In [10]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = DataSetV1(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, 
        num_workers=0 
        )
    
    return dataloader


In [11]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  262,  2769,  3211,    12],
        [12036,    13,   843,   523],
        [  616,   705, 23873,  2350],
        [ 2588,   856,   607,  2781],
        [  198,  1544, 13818,  4622],
        [   62,   438,   270,   561],
        [  319,  3619,   438,   505],
        [  286,   262,  5977,   373]])

Targets:
 tensor([[ 2769,  3211,    12, 49655],
        [   13,   843,   523,   438],
        [  705, 23873,  2350,     6],
        [  856,   607,  2781,   526],
        [ 1544, 13818,  4622,    11],
        [  438,   270,   561,   423],
        [ 3619,   438,   505,   286],
        [  262,  5977,   373, 29178]])


Now data is converted token and we need to convert to embedding so we can get meaning of the word/token. for example we can consider embedding dimensions as feature and value in the embedding shows is that token has that feature. Also we will add positional embedding to understand where the token appear in the context. Initially those embedding values are random and trained to proper value through neural network backpropogation.

There are two types of positional encoding: absolute and relative.

Absolute positional encoding assigns a unique embedding to each position in the sequence (for example, position 1, position 2, and so on). As the name suggests, it represents the exact position of a token. In many modern models, these positional embeddings are learned during training rather than remaining fixed.

Relative positional encoding, on the other hand, represents the distance between tokens instead of their absolute positions. It captures how far one token is from another, allowing the model to better generalize across sequences with different context lengths.

In [18]:
max_length = 4
vocab_size = 50257
output_dim = 256

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
token_embeddings = embedding_layer(inputs)
# print(token_embeddings)

context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))

print(pos_embeddings)

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings)


tensor([[-0.3457,  0.9631,  0.6797,  ...,  1.6313, -1.1621,  1.0640],
        [ 0.5489, -0.3957,  0.0949,  ...,  0.9179,  1.1409,  0.9334],
        [ 0.1617, -0.5287, -0.0738,  ...,  0.1901, -1.5437,  1.9048],
        [ 0.2118, -0.4736, -3.2282,  ..., -1.0637, -2.9734,  0.4224]],
       grad_fn=<EmbeddingBackward0>)
tensor([[[-1.7518,  0.3229, -1.0236,  ...,  2.3096, -1.9589,  1.7285],
         [ 1.3279,  0.3982,  0.0911,  ..., -1.1309,  1.5384,  1.7831],
         [ 1.2968, -2.1671,  1.0230,  ..., -1.4245, -2.0687,  1.7731],
         [-0.1069, -1.8811, -3.3252,  ..., -0.5042, -3.9723, -0.5210]],

        [[ 1.6571,  2.0962,  1.2647,  ...,  0.5953, -1.9965,  1.7791],
         [ 0.4172,  0.4649,  1.6533,  ...,  0.3804,  1.6742,  1.6626],
         [-0.1130, -0.2035,  0.0363,  ...,  0.3521, -0.6996,  2.9998],
         [ 1.2466, -1.1653, -3.3082,  ...,  0.6956, -2.2045,  0.3002]],

        [[ 0.7645,  1.5529,  1.1136,  ...,  1.8163, -0.6602,  2.5681],
         [-0.6109,  0.1848,  0.4394,  .